In [ ]:
# ========== 导入：环境、压缩、类型标注、Gradio、OpenAI ==========

# 标准库 os：读 OPENAI_API_KEY 等环境变量
import os
# io.BytesIO：在内存里拼 zip，不必先落临时文件
import io
# zipfile：把策略 / broker / backtest / README 打成可下载包
import zipfile
# textwrap：本笔记本导入但下方未强制使用（保持原 import）
import textwrap
# typing：Dict / Generator / List 等类型标注，方便读签名
from typing import Dict, Generator, List, Optional, Tuple
# gradio：多 Tab 小工具 UI（Docstrings / Tests / Trading Scaffold）
import gradio as gr
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI 客户端：云端官方 API + 本地 Ollama 的 OpenAI 兼容端点各一个
from openai import OpenAI


In [ ]:
# ========== 双客户端：OpenAI 云端 + 本地 Ollama（OpenAI 兼容） ==========

# 加载 .env（默认不 override；参数保持原样）
load_dotenv()
# 云端客户端：密钥来自 OPENAI_API_KEY
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# 本地 Ollama：走 /v1 兼容接口；api_key 占位字符串 'ollama'（Ollama 通常不校验）
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key = 'ollama')


In [ ]:
# ========== 模型下拉选项：openai/* 与 ollama/* 前缀路由 ==========

# 带路由前缀的选项；真正调用前会 normalize 去掉前缀
MODEL_CHOICES = [
    "openai/gpt-4o-mini",
    "openai/gpt-4o",
    "openai/gpt-3.5-turbo",
    "ollama/llama3.2",
    "ollama/phi3:mini",
    "ollama/qwen2.5:3b",
]
# 默认选中的选项（保持原样）
DEFAULT_MODEL = "openai/gpt-4o-mini"


In [ ]:
# ========== 路由与流式：按前缀选客户端，边收边 yield 累积文本 ==========

def normalize_model(model: str) -> str:
    # "openai/gpt-4o-mini" → "gpt-4o-mini"；只拆一次，保留模型名里可能的 '/'
    return model.split("/", 1)[1]

def get_client(model_choice: str) -> OpenAI:
    # ollama/ 前缀走本地兼容端点，否则走云端 client
    if model_choice.startswith("ollama/"):
        return ollama_via_openai
    return client

def stream_response(model_choice: str, messages: List[Dict]) -> Generator[str, None, None]:
    # 去掉路由前缀，得到 API 真正要的 model 名
    model_name = normalize_model(model_choice)
    # 按前缀拿到对应客户端
    api_client = get_client(model_choice)

    try:
        # stream=True：逐 chunk 推送；temperature / max_tokens 保持原参数
        stream = api_client.chat.completions.create(
            model=model_name,
            messages=messages,
            stream=True,
            max_tokens=1500,
            temperature=0.3,
        )
        # 累积已生成文本；每次 yield 全文，方便 Gradio 渐进刷新
        text = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            text += delta
            yield text
    except Exception as e:
        # 流式失败时 yield 错误说明（文案保持原样）
        yield f"Error while streaming from {model_choice}: {str(e)}"


In [ ]:
# ========== Prompt 模板：system 定角色；DOC / TEST 任务说明保持英文 ==========

# 总 system：偏保守的资深 Python 工程师人设（影响回答风格，不翻译）
SYSTEM_PROMPT = """You are a senior Python engineer. Be conservative and helpful.
When asked to annotate code, add clean Google-style docstrings and minimal comments.
When asked to generate tests, write readable pytest test modules.
When asked to create a trading scaffold, make a working 3-file setup with backtesting.
Avoid clever tricks, prioritize clarity and correctness."""

# Docstring 任务模板：{style} / {add_types} 运行时 format 填入
DOC_PROMPT = """Task: Add docstrings and helpful inline comments to this code.
Do not change any logic or flow.
Use {style}-style docstrings. Add type hints: {add_types}.
Return only the updated code."""

# 单测任务模板：要求 pytest、覆盖正常/边界/错误
TEST_PROMPT = """Task: Generate a pytest test file for this code.
Include tests for normal, edge, and error conditions.
Use plain pytest (no unittest). Add minimal mocks if needed."""


In [ ]:
# ========== 交易脚手架：本地拼出 strategy / broker / backtest / README（不调 LLM） ==========

def make_trading_files(strategy_brief: str, symbol: str) -> Dict[str, str]:
    """Return dictionary of files for a simple strategy, broker, and backtest."""
    # strategy.py：示例 SMA 交叉；brief / symbol 写入模块头说明
    strategy_py = f'''"""
strategy.py
Auto-generated strategy module for {symbol}.
Brief: {strategy_brief}
"""
def decide(state, bar):
    """Example SMA crossover."""
    prices = state.setdefault("prices", [])
    prices.append(bar["close"])
    if len(prices) < 20:
        return "HOLD", state
    short = sum(prices[-5:]) / 5
    long = sum(prices[-20:]) / 20
    action = "BUY" if short > long and not state.get("pos") else "SELL" if short < long and state.get("pos") else "HOLD"
    state["pos"] = action == "BUY" or (state.get("pos") and action != "SELL")
    return action, state
'''

    # sim_broker.py：内存券商，买卖改 cash/pos，记录 equity 与 trades
    broker_py = """\"\"\"sim_broker.py
Simple in-memory broker simulator.\"\"\"
def init(cash=10000.0):
    return {"cash": cash, "pos": 0, "equity": [], "trades": []}

def execute(state, action, price, size=1):
    if action == "BUY" and state["cash"] >= price * size:
        state["cash"] -= price * size
        state["pos"] += size
        state["trades"].append(("BUY", price))
    elif action == "SELL" and state["pos"] >= size:
        state["cash"] += price * size
        state["pos"] -= size
        state["trades"].append(("SELL", price))
    state["equity"].append(state["cash"] + state["pos"] * price)
    return state
"""

    # backtest.py：合成随机行情，循环 decide → execute，打印最终权益
    backtest_py = f'''"""
backtest.py
Run a synthetic backtest for {symbol}.
"""
import random, strategy, sim_broker

def synthetic_data(n=250, start=100.0):
    price = start
    data = []
    for _ in range(n):
        price *= 1 + random.uniform(-0.01, 0.01)
        data.append({{"close": price}})
    return data

def run():
    bars = synthetic_data()
    state = {{"prices": []}}
    broker = sim_broker.init()
    for bar in bars:
        action, state = strategy.decide(state, bar)
        broker = sim_broker.execute(broker, action, bar["close"])
    eq = broker["equity"][-1] if broker["equity"] else broker["cash"]
    print(f"Final equity: {{eq:.2f}} | Trades: {{len(broker['trades'])}}")

if __name__ == "__main__":
    run()
'''

    # README：说明三文件用途与运行方式
    readme = f"""# Trading Scaffold for {symbol}
Generated from your brief: {strategy_brief}
Files:
- strategy.py — core logic
- sim_broker.py — in-memory execution
- backtest.py — synthetic backtest
Run with:
```bash
python backtest.py
```"""
    # 文件名 → 内容；供预览与打 zip
    return {
        "strategy.py": strategy_py,
        "sim_broker.py": broker_py,
        "backtest.py": backtest_py,
        "README.md": readme,
    }


In [ ]:
# ========== 打包：把脚手架文件写入内存 zip，返回文件名与字节 ==========

def zip_trading_files(files: Dict[str, str]) -> Tuple[str, bytes]:
    # 内存缓冲区当「虚拟文件」
    buf = io.BytesIO()
    # ZIP_DEFLATED：压缩写入每个成员
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for name, content in files.items():
            z.writestr(name, content)
    # 读指针回到开头，取出全部字节
    buf.seek(0)
    return "trading_scaffold.zip", buf.read()


In [ ]:
# ========== Tab1：流式生成 docstring / 注释（不改逻辑） ==========

def docstrings_stream(model_choice: str, code: str, style: str, add_types: bool):
    # 空代码：提示粘贴（文案保持原样）
    if not code.strip():
        yield "Please paste Python code first."
        return
    # system + user（DOC_PROMPT.format + 源码）
    sys = {"role": "system", "content": SYSTEM_PROMPT}
    usr = {
        "role": "user",
        "content": DOC_PROMPT.format(style=style, add_types=add_types) + "\n\n" + code,
    }
    # 把流式累积文本一路 yield 给 Gradio Markdown
    yield from stream_response(model_choice, [sys, usr])


In [ ]:
# ========== Tab2：流式生成 pytest 单测文件 ==========

def tests_stream(model_choice: str, code: str):
    # 空代码直接提示
    if not code.strip():
        yield "Please paste Python code first."
        return
    # system 用人设；user 用 TEST_PROMPT + 源码
    sys = {"role": "system", "content": SYSTEM_PROMPT}
    usr = {"role": "user", "content": TEST_PROMPT + "\n\n" + code}
    yield from stream_response(model_choice, [sys, usr])


In [ ]:
# ========== Tab3：生成交易脚手架并落盘 zip，供下载与预览 ==========

def trading_scaffold(strategy_brief: str, symbol: str):
    # 空 symbol 默认 AAPL（默认值保持原样）
    if not symbol.strip():
        symbol = "AAPL"
    # brief 为空时用默认 SMA 说明
    files = make_trading_files(strategy_brief or "Simple SMA crossover", symbol)
    # 打成 zip 字节
    name, data = zip_trading_files(files)
    zip_path = "trading_scaffold.zip"
    # 写到当前工作目录，供 gr.File 下载
    with open(zip_path, "wb") as f:
        f.write(data)
    # 返回四段预览文本 + zip 路径（对应 Gradio 五个输出）
    return (
        files["strategy.py"],
        files["sim_broker.py"],
        files["backtest.py"],
        files["README.md"],
        zip_path,
    )


In [ ]:
# ========== Gradio Blocks：三个 Tab（Docstrings / Tests / Trading Scaffold） ==========

with gr.Blocks(title="DevLab Assistant") as demo:
    # 标题与总说明（UI 字符串保持原样）
    gr.Markdown("# DevLab Assistant")
    gr.Markdown(
        "This mini-lab helps with everyday coding tasks:\n"
        "* Add docstrings and helpful comments to existing code\n"
        "* Generate pytest unit tests automatically\n"
        "* Scaffold a small trading simulator to experiment with strategy ideas\n\n"
        "Select a model (OpenAI or a local Ollama one) and try each tab."
    )

    with gr.Tab("Docstrings / Comments"):
        # 模型、docstring 风格、是否加类型标注
        model1 = gr.Dropdown(MODEL_CHOICES, value=DEFAULT_MODEL, label="Model")
        style = gr.Radio(["google", "numpy"], value="google", label="Docstring style")
        add_types = gr.Checkbox(value=False, label="Add basic type hints")
        code_input = gr.Textbox(
            lines=14,
            label="Paste your Python code here",
            placeholder="def multiply(a, b):\n    return a * b",
        )
        output_md = gr.Markdown(label="Result")
        gen_btn = gr.Button("Generate Docstrings")

        # 点击后流式填充 Markdown
        gen_btn.click(
            fn=docstrings_stream,
            inputs=[model1, code_input, style, add_types],
            outputs=output_md,
        )

    with gr.Tab("Unit Tests"):
        model2 = gr.Dropdown(MODEL_CHOICES, value=DEFAULT_MODEL, label="Model")
        code_input2 = gr.Textbox(
            lines=14,
            label="Paste the code you want tests for",
            placeholder="class Calculator:\n    def add(self, a, b):\n        return a + b",
        )
        output_md2 = gr.Markdown(label="Generated test file")
        gen_btn2 = gr.Button("Generate Tests")

        gen_btn2.click(
            fn=tests_stream,
            inputs=[model2, code_input2],
            outputs=output_md2,
        )

    with gr.Tab("Trading Scaffold"):
        gr.Markdown(
            "Generate a minimal, self-contained trading simulator that includes:\n"
            "* `strategy.py`: basic SMA crossover strategy\n"
            "* `sim_broker.py`: in-memory broker\n"
            "* `backtest.py`: synthetic data backtest\n"
            "You can run it locally with `python backtest.py`."
        )

        brief = gr.Textbox(
            lines=6,
            label="Strategy Brief",
            placeholder="e.g., SMA crossover with fast=5, slow=20, long-only",
        )
        symbol = gr.Textbox(value="AAPL", label="Symbol")
        gen_btn3 = gr.Button("Generate Scaffold")

        # 四个预览组件 + 一个下载
        s_md = gr.Code(language="python", label="strategy.py")
        b_md = gr.Code(language="python", label="sim_broker.py")
        bt_md = gr.Code(language="python", label="backtest.py")
        r_md = gr.Markdown(label="README.md")
        zip_out = gr.File(label="Download ZIP")

        gen_btn3.click(
            fn=trading_scaffold,
            inputs=[brief, symbol],
            outputs=[s_md, b_md, bt_md, r_md, zip_out],
        )

    # 页脚提示：Ollama pull / OPENAI_API_KEY（文案保持原样）
    gr.Markdown("---")
    gr.Markdown(
        "Tips:\n"
        "* Ollama models must be pulled locally (for example `ollama pull llama3.2`).\n"
        "* OpenAI models require the `OPENAI_API_KEY` environment variable.\n"
        "* Everything runs safely and offline for the local models."
    )


In [ ]:
# ========== 入口：直接运行本文件时启动 Gradio ==========

# 作为脚本执行时 launch；在 Jupyter 里通常直接跑上一格 Blocks 定义即可
if __name__ == "__main__":
    demo.launch()
